# Nour Deep Learning Lab

This notebook uses Colab GPU/TPU compute when available and saves every checkpoint to Google Drive. It is designed to resume after a runtime disconnect.

The model is an educational deep classifier trained on the real Wisconsin Breast Cancer dataset. It is not a medical diagnostic system.

In [ ]:
# Check the runtime hardware
import torch, os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU assigned; change Runtime > Change runtime type > GPU.")

In [ ]:
# Persistent storage
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/NourML")
PROJECT.mkdir(parents=True, exist_ok=True)
CHECKPOINT = PROJECT / "deep_model_checkpoint.pt"
METRICS = PROJECT / "training_metrics.jsonl"
print("checkpoints:", PROJECT)

In [ ]:
# Data
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=.2, random_state=42, stratify=data.target)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32"); X_test = scaler.transform(X_test).astype("float32")
train = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.float32).view(-1,1)), batch_size=64, shuffle=True)
test_x=torch.tensor(X_test).to(device); test_y=torch.tensor(y_test,dtype=torch.float32).view(-1,1).to(device)

In [ ]:
# Four-hidden-layer deep network
import torch.nn as nn
model = nn.Sequential(nn.Linear(30,128), nn.ReLU(), nn.Dropout(.10), nn.Linear(128,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,16), nn.ReLU(), nn.Linear(16,1)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()
start_epoch=0
if CHECKPOINT.exists():
    saved=torch.load(CHECKPOINT, map_location=device)
    model.load_state_dict(saved["model"]); optimizer.load_state_dict(saved["optimizer"]); start_epoch=saved["epoch"]+1
    print("resumed from epoch", start_epoch)
else: print("starting a new model")

In [ ]:
# Train and checkpoint
import json, time
for epoch in range(start_epoch, start_epoch+25):
    model.train(); total=0
    for xb,yb in train:
        xb,yb=xb.to(device),yb.to(device); optimizer.zero_grad(); logits=model(xb); loss=loss_fn(logits,yb); loss.backward(); optimizer.step(); total += loss.item()
    model.eval()
    with torch.no_grad():
        prob=torch.sigmoid(model(test_x)); pred=(prob>=.5).float(); acc=(pred==test_y).float().mean().item()
    row={"epoch":epoch,"loss":total/len(train),"test_accuracy":acc,"time":time.time()}
    with METRICS.open("a") as f: f.write(json.dumps(row)+"\n")
    torch.save({"epoch":epoch,"model":model.state_dict(),"optimizer":optimizer.state_dict(),"scaler_mean":scaler.mean_,"scaler_scale":scaler.scale_}, CHECKPOINT)
    if epoch%5==0 or epoch==start_epoch+24: print(row)
print("saved:", CHECKPOINT)

In [ ]:
# Inspect the result
model.eval()
with torch.no_grad():
    accuracy=((torch.sigmoid(model(test_x))>=.5)==test_y).float().mean().item()
print(f"final test accuracy: {accuracy:.1%}")

## How to use

1. Open this file in Google Colab.
2. Choose **Runtime → Change runtime type → GPU**.
3. Run the cells from top to bottom.
4. Leave the checkpoint in Google Drive.
5. On a later session, rerun the notebook; it resumes from the latest checkpoint.

Colab can disconnect, so Drive checkpoints are part of the design.